# 02 Data Validation

This notebook validates schema coverage, split sizes, edge-case rate, required keys, and distribution sanity.

In [1]:
from __future__ import annotations
import json
import logging
import re
from pathlib import Path
from typing import Any, Dict, List
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(levelname)s:%(message)s')
logger = logging.getLogger('data_validation')
ROOT = Path.cwd()
schema = json.loads((ROOT / 'schema.json').read_text(encoding='utf-8'))
splits = {name: json.loads((ROOT / f'{name}.json').read_text(encoding='utf-8')) for name in ['train', 'validation', 'test']}

In [2]:
def flatten_records(splits: Dict[str, List[Dict[str, Any]]]) -> pd.DataFrame:
    rows = []
    for split, records in splits.items():
        for record in records:
            rows.append({**record, 'split': split})
    return pd.DataFrame(rows)

def required_slot_map(schema: Dict[str, Any]) -> Dict[str, List[str]]:
    return {intent['name']: [slot['name'] for slot in intent['slots'] if slot['required']] for intent in schema['intents']}

def validate_record(record: Dict[str, Any], valid_intents: set[str]) -> List[str]:
    errors = []
    for key in ['sample_id', 'intent', 'slots', 'customer_utterance', 'target_text', 'edge_case', 'edge_case_types', 'metadata']:
        if key not in record: errors.append(f'missing top-level key: {key}')
    if record.get('intent') not in valid_intents: errors.append('unknown intent')
    if not isinstance(record.get('slots'), dict): errors.append('slots must be an object')
    return errors

valid_intents = {intent['name'] for intent in schema['intents']}
validation_errors = [(record.get('sample_id'), validate_record(record, valid_intents)) for records in splits.values() for record in records]
validation_errors = [(sid, errors) for sid, errors in validation_errors if errors]
assert not validation_errors, validation_errors[:5]
logger.info('All records passed structural validation')

INFO:All records passed structural validation


In [3]:
df = flatten_records(splits)
assert len(df) == 400
assert len(splits['train']) == 280 and len(splits['validation']) == 60 and len(splits['test']) == 60
assert df['edge_case'].mean() >= 0.25
intent_counts = df['intent'].value_counts(normalize=True)
edge_counts = df.explode('edge_case_types')['edge_case_types'].value_counts(dropna=True)
display(intent_counts.to_frame('observed_share'))
display(edge_counts.to_frame('edge_case_count'))

,observed_share
intent,
order_status,0.1225
delivery_delay,0.1200
return_item,0.0975
cancel_order,0.0925
product_availability,0.0925
refund_status,0.0750
modify_order,0.0750
complaint_registration,0.0725
payment_issue,0.0625


,edge_case_count
edge_case_types,
partial_information,31
empty_value,27
duplicate_entities,26
null_value,23
invalid_id,21
missing_required_slot,17
multiple_products,17
ambiguous_request,17


In [4]:
def is_invalid_id(value: Any) -> bool:
    if not isinstance(value, str): return False
    return bool(value in {'12345', 'ORD', 'ORDER-??', 'INVALID_ID', 'TXN-ABC'})

def summarize_edge_signals(records: List[Dict[str, Any]]) -> Dict[str, int]:
    summary = {'null_values': 0, 'empty_values': 0, 'invalid_ids': 0, 'multi_product_rows': 0, 'duplicate_product_rows': 0}
    for record in records:
        values = list((record.get('slots') or {}).values())
        summary['null_values'] += sum(value is None for value in values)
        summary['empty_values'] += sum(value == '' or value == [] for value in values)
        summary['invalid_ids'] += sum(is_invalid_id(value) for value in values)
        products = (record.get('slots') or {}).get('products')
        if isinstance(products, list) and len(products) > 1: summary['multi_product_rows'] += 1
        if isinstance(products, list) and len(products) != len(set(products)): summary['duplicate_product_rows'] += 1
    return summary

edge_summary = summarize_edge_signals(df.to_dict('records'))
display(pd.Series(edge_summary).to_frame('count'))
assert edge_summary['null_values'] > 0
assert edge_summary['empty_values'] > 0
assert edge_summary['invalid_ids'] > 0

,count
null_values,23
empty_values,26
invalid_ids,17
multi_product_rows,131
duplicate_product_rows,23
